# Chess Transformer — Kaggle Training (resumible)
Pipeline con agentes LLM (NVIDIA) + resume + AMP + val. Si la sesión muere, re-ejecuta desde la celda de train: continúa desde `checkpoints/last.pt`.


In [ ]:
!git clone https://github.com/oscar2697/chess-transformer.git 2>/dev/null; true
%cd chess-transformer
!git pull origin main
!git log --oneline -2
!ls src/graph_agents/


In [ ]:
!pip install python-chess torch matplotlib langchain-openai -q
from src.graph_agents.llm_agents import run_agent, get_llm
print('import ok, backend:', get_llm()[0])


In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['NVIDIA_API_KEY'] = UserSecretsClient().get_secret('NVIDIA_KEY')
os.environ['LLM_MODEL'] = 'moonshotai/kimi-k3'
from src.graph_agents.llm_agents import get_llm
print('backend:', get_llm()[0])  # esperado: nvidia


In [ ]:
!mkdir -p data/raw
!wget -q https://database.nikonoel.fr/lichess_elite_2024-01.zip -O data/raw/elite_2024-01.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-02.zip -O data/raw/elite_2024-02.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-03.zip -O data/raw/elite_2024-03.zip
!unzip -o -q data/raw/elite_2024-01.zip -d data/raw/
!unzip -o -q data/raw/elite_2024-02.zip -d data/raw/
!unzip -o -q data/raw/elite_2024-03.zip -d data/raw/
!ls -lh data/raw/*.pgn


In [ ]:
from src.graph_agents.llm_agents import run_agent
r = run_agent('data_engineer', 'Preprocess 3 elite months',
              pgn_path=['data/raw/lichess_elite_2024-01.pgn',
                        'data/raw/lichess_elite_2024-02.pgn',
                        'data/raw/lichess_elite_2024-03.pgn'],
              elo_threshold=2000, max_positions=600000)
print('mode:', r['mode'])
print(r['result'])


In [ ]:
# CELDA RE-EJECUTABLE: si muere la sesión, corre de nuevo y continúa desde last.pt
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Scale training', epochs=15, batch_size=128, seed=42,
              resume=True, use_amp=True)
print('mode:', r['mode'])
print(r['result'])


In [ ]:
from src.graph_agents.llm_agents import run_agent
print(run_agent('evaluator', 'Evaluate policy/value vs Stockfish')['result'])
print(run_agent('writer', 'Write results tables')['result'][:300])


In [ ]:
!tar -czf artifacts.tar.gz checkpoints/best_model.pt checkpoints/last.pt experiments/ paper/main.tex data/processed/stats.json data/processed/vocab.json
!ls -lh artifacts.tar.gz
# Descarga artifacts.tar.gz + data/processed/train.jsonl + val.jsonl desde el panel Output